# Lecture 04 - Solutions to Part B (Python Applications)

Runnable Python solutions for Exercises 18-24. The by-hand solutions for Part A (Exercises 1-17) are provided separately in `Lecture_04_solutions_by_hand.md`.

All returns in the data file are in **decimal** form (e.g. 0.021 = 2.1%), so a "-1% market return" threshold is `-0.01` and "-2% stock return" is `-0.02`. By default `DATA_DIRECTORY = "../data/"`, matching the standard layout (notebook in `lecture_files/`, data in a sibling `data/` folder). If you keep the data files in a different directory, change `DATA_DIRECTORY` to point there (for example `"data and notebooks/"`).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Location of the data files. In the standard course layout this notebook lives
# in "lecture_files/" and the data in a sibling "data/" folder, so the default
# points there. Change it if your data is elsewhere (e.g. "./" if same folder).
DATA_DIRECTORY = "../data/"
CSV = Path(DATA_DIRECTORY) / "Lecture 04 market loss days.csv"

df = pd.read_csv(CSV)
n = len(df)
print("shape:", df.shape, "| columns:", list(df.columns))
df.head()

shape: (180, 3) | columns: ['date', 'market_return', 'stock_return']


,date,market_return,stock_return
0,2024-01-02,0.021185,0.011540
1,2024-01-03,-0.008034,-0.043358
2,2024-01-04,-0.012135,-0.040573
3,2024-01-05,-0.004272,-0.022414
4,2024-01-08,-0.012905,-0.020074


## Exercise 18 - Build Event Variables from Returns

In [2]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df["market_stress"]     = df["market_return"] < -0.01     # market return below -1%
df["stock_loss"]        = df["stock_return"]  < 0          # any stock loss
df["large_stock_loss"]  = df["stock_return"]  < -0.02     # stock return below -2%
df["same_direction"]    = np.sign(df["market_return"]) == np.sign(df["stock_return"])

df.head()

,date,market_return,stock_return,market_stress,stock_loss,large_stock_loss,same_direction
0,2024-01-02,0.021185,0.011540,False,False,False,True
1,2024-01-03,-0.008034,-0.043358,False,True,True,True
2,2024-01-04,-0.012135,-0.040573,True,True,True,True
3,2024-01-05,-0.004272,-0.022414,False,True,True,True
4,2024-01-08,-0.012905,-0.020074,True,True,True,True


In [3]:
events = ["market_stress","stock_loss","large_stock_loss","same_direction"]
rel_freq = df[events].mean().rename("relative_frequency")
print(rel_freq.round(4))

market_stress       0.1833
stock_loss          0.4833
large_stock_loss    0.2000
same_direction      0.6944
Name: relative_frequency, dtype: float64


**Relative frequencies:** market stress 0.183 (33/180), any stock loss 0.483 (87/180), large stock loss 0.200 (36/180), same-direction days 0.694 (125/180).

**Step 6.** Each figure is first a *descriptive statistic* - the fraction of these 180 observed days on which the event happened. It becomes a *probability estimate* only under the extra assumption that this sample is representative of the future days we want to reason about. The number is the same; the interpretation (what happened vs. what may happen) is a modelling choice.

## Exercise 19 - Probability Rules from Event Flags

In [4]:
A = df["market_stress"]          # event A
B = df["large_stock_loss"]       # event B

PA   = A.mean()
PAc  = 1 - PA
PB   = B.mean()
PAB  = (A & B).mean()
PAuB = (A | B).mean()

print(f"P(A)      = {PA:.4f}   ({A.sum()}/{n})")
print(f"P(A^c)    = {PAc:.4f}")
print(f"P(B)      = {PB:.4f}   ({B.sum()}/{n})")
print(f"P(A n B)  = {PAB:.4f}   ({(A&B).sum()}/{n})")
print(f"P(A u B)  = {PAuB:.4f}   ({(A|B).sum()}/{n})")
print(f"check P(A)+P(B)-P(A n B) = {PA+PB-PAB:.4f}")

P(A)      = 0.1833   (33/180)
P(A^c)    = 0.8167
P(B)      = 0.2000   (36/180)
P(A n B)  = 0.0944   (17/180)
P(A u B)  = 0.2889   (52/180)
check P(A)+P(B)-P(A n B) = 0.2889


**Results.** \(P(A)=0.183,\ P(A^c)=0.817,\ P(B)=0.200,\ P(A\cap B)=0.094\) (17/180), \(P(A\cup B)=0.289\) (52/180). The union identity holds exactly: \(0.183+0.200-0.094=0.289\).

**Interpretation (step 7).** On about **29%** of the days in this sample there was *either* market stress *or* a large stock loss (or both).

## Exercise 20 - Conditional Probability and Risk Lift

In [5]:
PB_given_A  = B[A].mean()
PB_given_Ac = B[~A].mean()
lift = PB_given_A / PB_given_Ac
print(f"P(B|A)   = {PB_given_A:.4f}   ({(A&B).sum()}/{A.sum()})   large loss on stress days")
print(f"P(B|A^c) = {PB_given_Ac:.4f}   ({(~A&B).sum()}/{(~A).sum()})   large loss on calm days")
print(f"risk lift = P(B|A)/P(B|A^c) = {lift:.2f}")

P(B|A)   = 0.5152   (17/33)   large loss on stress days
P(B|A^c) = 0.1293   (19/147)   large loss on calm days
risk lift = P(B|A)/P(B|A^c) = 3.99


**Results.** \(P(B\mid A)=0.515\) (17/33) versus \(P(B\mid A^c)=0.129\) (19/147), a **risk lift of about 4.0**.

**Interpretation (step 4).** A large stock loss is roughly four times as likely on a market-stress day as on a calm day: about 52% versus 13%.

**Step 5.** This is an *association* in the historical sample, not a proven cause. Both events are plausibly driven by the same underlying market factors; the numbers show they move together, but the data alone do not establish that stress *causes* the stock's large losses.

## Exercise 21 - Independence Benchmark

In [6]:
observed_both = (A & B).sum()
expected_both = n * PA * PB          # count expected if A, B were independent
print(f"observed joint days           = {observed_both}")
print(f"expected under independence    = {expected_both:.2f}   (= n*P(A)*P(B))")
print(f"ratio observed/expected        = {observed_both/expected_both:.2f}")

observed joint days           = 17
expected under independence    = 6.60   (= n*P(A)*P(B))
ratio observed/expected        = 2.58


**Results.** Observed joint days = **17**; expected under independence = \(180\times0.183\times0.200\approx 6.6\).

**Interpretation (steps 3-4).** The two events co-occur about **2.6 times more often** than independence predicts, so the sample clearly does **not** behave as if market stress and large stock losses were independent - consistent with the risk lift in Exercise 20.

**Step 5.** Independence is a strong statement: it requires \(P(A\cap B)=P(A)P(B)\) *exactly*, i.e. that conditioning on one event leaves the other's probability unchanged. Merely being able to occur together is much weaker - almost any two events can co-occur; independence is about whether one carries *no information* about the other.

## Exercise 22 - Build a Discrete Return-State Distribution

In [7]:
def classify(r):
    if r < 0:      return "Loss"
    if r < 0.01:   return "Flat or small gain"
    return "Strong gain"

df["stock_state"] = df["stock_return"].apply(classify)
order = ["Loss","Flat or small gain","Strong gain"]
counts = df["stock_state"].value_counts().reindex(order)
probs  = counts / n
dist = pd.DataFrame({"count": counts, "probability": probs.round(4)})
print(dist)
print("sum of probabilities:", round(probs.sum(), 4))

                    count  probability
stock_state                           
Loss                   87       0.4833
Flat or small gain     39       0.2167
Strong gain            54       0.3000
sum of probabilities: 1.0


**Distribution.** Loss 87 (0.483), Flat or small gain 39 (0.217), Strong gain 54 (0.300); the probabilities sum to 1.

**Step 6.** As a *description* of these 180 days the table is just observed relative frequencies. The moment we use it to describe an as-yet-unobserved future trading day, it becomes a **model**: we are assuming the same three-state mechanism, with the same probabilities, will govern the next day - an assumption that holds only if the process is stable.

## Exercise 23 - Expected Payoff from an Empirical Distribution

In [8]:
df["strategy_payoff"] = np.where(df["large_stock_loss"], -0.03, 0.004)   # -3.0% vs +0.4%

E_payoff  = df["strategy_payoff"].mean()
SD_payoff = df["strategy_payoff"].std(ddof=1)
E_stress    = df.loc[A,  "strategy_payoff"].mean()
E_no_stress = df.loc[~A, "strategy_payoff"].mean()

print(f"empirical E[payoff]        = {E_payoff:.5f}  = {E_payoff*100:.3f}%")
print(f"empirical SD[payoff]       = {SD_payoff:.5f}  = {SD_payoff*100:.3f}%")
print(f"E[payoff | market stress]  = {E_stress*100:.3f}%")
print(f"E[payoff | no stress]      = {E_no_stress*100:.3f}%")

empirical E[payoff]        = -0.00280  = -0.280%
empirical SD[payoff]       = 0.01364  = 1.364%
E[payoff | market stress]  = -1.352%
E[payoff | no stress]      = -0.039%


**Results.** Overall empirical expected payoff \(\approx -0.28\%\) with a standard deviation of about \(1.36\%\). Split by regime it is about \(-1.35\%\) on market-stress days and \(-0.04\%\) on calm days.

The strategy loses money on average because the \(-3\%\) hit on the 36 large-loss days outweighs the small \(+0.4\%\) collected on the other days, and the damage is concentrated in the stress regime.

**Step 5.** This expected payoff is a **sample average**, not a promise about tomorrow. Any single day returns either \(-3\%\) or \(+0.4\%\); the average simply summarises the historical mix of the two, and would change if the frequency of large-loss days changed.

## Exercise 24 - Careful Probability Memo

In [9]:
period = f'{df["date"].min().date()} to {df["date"].max().date()}'
print("Key figures for the memo")
print("-"*40)
print(f"sample: n = {n} trading days, {period}")
print(f"P(A) market stress            = {PA:.3f}")
print(f"P(B) large stock loss         = {PB:.3f}")
print(f"P(A n B)                      = {PAB:.3f}")
print(f"P(A u B)                      = {PAuB:.3f}")
print(f"P(B|A)                        = {PB_given_A:.3f}")
print(f"P(B|A^c)                      = {PB_given_Ac:.3f}")
print(f"observed vs expected joint    = {observed_both} vs {expected_both:.1f}")
print(f"empirical E[payoff]           = {E_payoff*100:.3f}%")

Key figures for the memo
----------------------------------------
sample: n = 180 trading days, 2024-01-02 to 2024-09-09
P(A) market stress            = 0.183
P(B) large stock loss         = 0.200
P(A n B)                      = 0.094
P(A u B)                      = 0.289
P(B|A)                        = 0.515
P(B|A^c)                      = 0.129
observed vs expected joint    = 17 vs 6.6
empirical E[payoff]           = -0.280%


**Memo - market stress and large stock losses**

*Sample.* This analysis uses **180 trading days** (2024-01-02 to 2024-09-09). All statistics below are sample estimates from this period.

*Definitions.* A day is a **market-stress** day (event \(A\)) when the market return is below \(-1\%\), and a **large-stock-loss** day (event \(B\)) when the stock return is below \(-2\%\).

*Probabilities.* \(P(A)=0.18\), \(P(B)=0.20\), \(P(A\cap B)=0.09\), and \(P(A\cup B)=0.29\): on roughly three days in ten there was either market stress or a large stock loss.

*Conditioning.* The large-loss rate is \(P(B\mid A)=0.52\) on stress days against \(P(B\mid A^c)=0.13\) on calm days - about a fourfold increase, so market stress is strongly informative about large stock losses.

*Independence.* The events are **not** independent in this sample: 17 joint days were observed against about 6.6 expected under independence.

*Strategy payoff.* The stylised strategy in Exercise 23 has an empirical expected payoff of about \(-0.28\%\) per day, with the losses concentrated on stress days.

*Limitation.* Every figure here is an estimate from one historical sample. It describes what happened over these 180 days and is **not** a guarantee about future days; if market conditions change, the probabilities and the strategy's payoff can change with them.